In [ ]:
#  Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.utils import shuffle
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

SEED = 13
N_SPLITS = 5

DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/TFM/DataSet")
INPUT_PATH = DATA_DIR / "01_only_text.csv"
PARTITIONS_PATH = DATA_DIR / "data_partitions_paper_ready.csv"
OUT_DIR = DATA_DIR / "02_results_text_xgboost_paper"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_PATH)
print("Partitions:", PARTITIONS_PATH)
print("Output:", OUT_DIR)

Input: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/01_only_text.csv
Partitions: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/data_partitions_paper_ready.csv
Output: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/02_results_text_xgboost_paper


In [ ]:
def get_metrics(y_true, y_pred, y_prob):
    return {
        "WAcc": accuracy_score(y_true, y_pred),
        "UAcc": balanced_accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


def evaluate_subject_level(pred_conv):
    # Media de probabilidades por sujeto 
    pred_subject = (
        pred_conv
        .groupby(["subject_id", "label", "outer_fold"], as_index=False)["prob_1"]
        .mean()
    )
    pred_subject["pred"] = (pred_subject["prob_1"] >= 0.5).astype(int)
    return pred_subject, get_metrics(
        pred_subject["label"],
        pred_subject["pred"],
        pred_subject["prob_1"],
    )

In [ ]:
data = pd.read_csv(INPUT_PATH)
partitions = pd.read_csv(PARTITIONS_PATH)

feature_cols = sorted(
    [col for col in data.columns if col.startswith("text_")],
    key=lambda col: int(col.split("_", 1)[1]),
)

required = {"subject_id", "avatar", "label"}
if not required.issubset(data.columns):
    raise ValueError(f"Faltan columnas en {INPUT_PATH.name}: {required - set(data.columns)}")
if not {"subject_id", "avatar", "outer_fold"}.issubset(partitions.columns):
    raise ValueError("data_partitions_paper_ready.csv debe tener subject_id, avatar y outer_fold")


partitions = shuffle(partitions, random_state=SEED).reset_index(drop=True)

df = data.merge(
    partitions[["subject_id", "avatar", "outer_fold"]],
    on=["subject_id", "avatar"],
    how="inner",
)

if len(df) != len(data):
    missing = len(data) - len(df)
    raise ValueError(f"Hay {missing} filas del CSV de embeddings sin partición externa")

print("Filas:", len(df))
print("Sujetos:", df["subject_id"].nunique())
print("Variables:", len(feature_cols))
print("Distribución de folds:")
display(df.groupby("outer_fold")["subject_id"].nunique().to_frame("n_subjects"))

Filas: 600
Sujetos: 101
Variables: 768
Distribución de folds:


,n_subjects
outer_fold,
1,21
2,20
3,20
4,20
5,20


In [5]:
param_grid = {
    "max_depth": list(range(3, 12)),
    "n_estimators": [25, 50, 100, 200],
}

scoring = {
    "WAcc": "accuracy",
    "UAcc": "balanced_accuracy",
    "auc": "roc_auc",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}

all_conv_predictions = []
conv_metrics_rows = []
subject_metrics_rows = []
best_params_rows = []

for fold in sorted(df["outer_fold"].unique()):
    print(f"===== OUTER FOLD {fold} =====")

    dev = df[df["outer_fold"] != fold].reset_index(drop=True)
    test = df[df["outer_fold"] == fold].reset_index(drop=True)

    X_dev = dev[feature_cols].to_numpy()
    y_dev = dev["label"].to_numpy()
    g_dev = dev["subject_id"].to_numpy()

    X_test = test[feature_cols].to_numpy()
    y_test = test["label"].to_numpy()

    inner_cv = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED,
    )

    grid = GridSearchCV(
        estimator=XGBClassifier(),
        param_grid=param_grid,
        scoring=scoring,
        refit="UAcc",
        cv=inner_cv,
        n_jobs=-1,
        verbose=0,
    )
    grid.fit(X_dev, y_dev, groups=g_dev)

    best_params = grid.best_params_
    best_idx = grid.best_index_
    best_inner_f1 = grid.cv_results_["mean_test_f1"][best_idx]
    best_inner_uacc = grid.cv_results_["mean_test_UAcc"][best_idx]

    model = XGBClassifier(**best_params)
    model.fit(X_dev, y_dev)

    prob_1 = model.predict_proba(X_test)[:, 1]
    pred = (prob_1 >= 0.5).astype(int)

    pred_conv = test[["subject_id", "avatar", "label", "outer_fold"]].copy()
    pred_conv["prob_1"] = prob_1
    pred_conv["pred"] = pred
    all_conv_predictions.append(pred_conv)

    conv_metrics = get_metrics(y_test, pred, prob_1)
    conv_metrics["outer_fold"] = fold
    conv_metrics_rows.append(conv_metrics)

    pred_subject, subject_metrics = evaluate_subject_level(pred_conv)
    subject_metrics["outer_fold"] = fold
    subject_metrics_rows.append(subject_metrics)

    best_params_rows.append({
        "outer_fold": fold,
        "best_max_depth": best_params["max_depth"],
        "best_n_estimators": best_params["n_estimators"],
        "best_inner_UAcc": best_inner_uacc,
        "best_inner_f1": best_inner_f1,
    })

    print("Best params:", best_params)
    print("Conversation-level F1:", round(conv_metrics["f1"], 3))
    print("Subject-level F1:", round(subject_metrics["f1"], 3))

===== OUTER FOLD 1 =====
Best params: {'max_depth': 7, 'n_estimators': 200}
Conversation-level F1: 0.348
Subject-level F1: 0.364
===== OUTER FOLD 2 =====
Best params: {'max_depth': 3, 'n_estimators': 200}
Conversation-level F1: 0.437
Subject-level F1: 0.571
===== OUTER FOLD 3 =====
Best params: {'max_depth': 5, 'n_estimators': 25}
Conversation-level F1: 0.482
Subject-level F1: 0.364
===== OUTER FOLD 4 =====
Best params: {'max_depth': 4, 'n_estimators': 50}
Conversation-level F1: 0.353
Subject-level F1: 0.182
===== OUTER FOLD 5 =====
Best params: {'max_depth': 4, 'n_estimators': 100}
Conversation-level F1: 0.463
Subject-level F1: 0.308


In [6]:
conv_metrics_df = pd.DataFrame(conv_metrics_rows)
subject_metrics_df = pd.DataFrame(subject_metrics_rows)
best_params_df = pd.DataFrame(best_params_rows)
predictions_df = pd.concat(all_conv_predictions, ignore_index=True)

global_subject_predictions, global_subject_metrics = evaluate_subject_level(predictions_df)

conv_metrics_df.to_csv(OUT_DIR / "conversation_level_outer_metrics.csv", index=False)
subject_metrics_df.to_csv(OUT_DIR / "subject_level_outer_metrics.csv", index=False)
best_params_df.to_csv(OUT_DIR / "best_params_by_outer_fold.csv", index=False)
predictions_df.to_csv(OUT_DIR / "conversation_predictions.csv", index=False)
global_subject_predictions.to_csv(OUT_DIR / "subject_predictions_global.csv", index=False)

print("Conversation-level Test metrics: mean ± std")
display(pd.concat([
    conv_metrics_df[["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]].mean().round(3).rename("mean"),
    conv_metrics_df[["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]].std().round(3).rename("std"),
], axis=1))

print("Subject-level Test metrics: mean ± std")
display(pd.concat([
    subject_metrics_df[["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]].mean().round(3).rename("mean"),
    subject_metrics_df[["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]].std().round(3).rename("std"),
], axis=1))

print("Subject-level global metrics")
display(pd.Series(global_subject_metrics).round(3).to_frame("global"))

print("Best params by outer fold")
display(best_params_df)

print("Archivos guardados en:", OUT_DIR)

Conversation-level Test metrics: mean ± std


,mean,std
WAcc,0.571,0.042
UAcc,0.542,0.043
auc,0.565,0.051
f1,0.417,0.062
precision,0.479,0.070
recall,0.371,0.068
kappa,0.087,0.089


Subject-level Test metrics: mean ± std


,mean,std
WAcc,0.623,0.069
UAcc,0.570,0.081
auc,0.631,0.127
f1,0.358,0.141
precision,0.640,0.277
recall,0.258,0.116
kappa,0.151,0.173


Subject-level global metrics


,global
WAcc,0.624
UAcc,0.572
auc,0.634
f1,0.367
precision,0.611
recall,0.262
kappa,0.156


Best params by outer fold


,outer_fold,best_max_depth,best_n_estimators,best_inner_UAcc,best_inner_f1
0,1,7,200,0.549980,0.427451
1,2,3,200,0.499630,0.334338
2,3,5,25,0.496535,0.303649
3,4,4,50,0.550536,0.378215
4,5,4,100,0.537204,0.389786


Archivos guardados en: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/02_results_text_xgboost_paper
